<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/02-congelar-e-soltar.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2. Congelar e soltar

Com poucas fotos, deixar a base inteira se mexer apaga o ofício. O desenho seguro é o contrário: a base fica quieta, a cabeça aprende o rótulo, e só depois o último bloco da base volta, com um passo curto.

Na ResNet esse último bloco se chama `layer4`. A primeira camada, `conv1`, continua travada. Ela já sabe achar borda e mancha. Não precisa reaprender isso com trinta fotos.

In [ ]:
!pip install -q torch torchvision

## Duas velocidades no mesmo passo

O otimizador aceita grupos. Cada grupo tem a sua taxa. A base que soltou anda a `1e-4`. A cabeça nova anda a `1e-3`, dez vezes mais. Passo grande na base desaprende. Passo grande na cabeça, que nasceu agora, é o que faz ela sair do zero.

In [ ]:
def contar(modelo, so_treinavel=False):
    total = 0
    for p in modelo.parameters():
        if so_treinavel and not p.requires_grad:
            continue
        total += p.numel()
    return total


import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(7)

modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
modelo.eval()
entrada = modelo.fc.in_features
for p in modelo.parameters():
    p.requires_grad = False
modelo.fc = nn.Linear(entrada, 2)
for p in modelo.layer4.parameters():
    p.requires_grad = True

grupo_base = [p for p in modelo.layer4.parameters() if p.requires_grad]
grupo_cabeca = list(modelo.fc.parameters())
otimizador = torch.optim.Adam([
    {"params": grupo_base, "lr": 1e-4},
    {"params": grupo_cabeca, "lr": 1e-3},
])

print(f"conv1 pode mudar? {modelo.conv1.weight.requires_grad}")
print(f"layer4 pode mudar? {all(p.requires_grad for p in modelo.layer4.parameters())}")
print(f"grupos de taxa: {len(otimizador.param_groups)}")
print(f"passo da base: {otimizador.param_groups[0]['lr']}")
print(f"passo da cabeça: {otimizador.param_groups[1]['lr']}")
print(f"pesos do layer4: {contar(modelo.layer4)}")

## Exercícios

Seis contas em cima do que já está na memória. Escreva a resposta na variável `resposta` e rode a célula: `conferir` diz na hora se fechou. Se travar, abra **Ver solução**.

In [ ]:
# @title Preparação dos exercícios (rode, não precisa ler)
VERIFICAR = {
    1: lambda r: bool(r) is False,
    2: lambda r: bool(r) is True,
    3: lambda r: int(r) == len(otimizador.param_groups),
    4: lambda r: abs(float(r) - otimizador.param_groups[0]["lr"]) < 1e-12,
    5: lambda r: abs(float(r) - otimizador.param_groups[1]["lr"]) < 1e-12,
    6: lambda r: int(r) == contar(modelo.layer4),
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

### Exercício 1. A porta de entrada

`conv1` é a primeira camada. Ela voltou a treinar? Responda `True` ou `False`.

<details>
<summary>Ver solução</summary>

```python
resposta = modelo.conv1.weight.requires_grad
```

</details>

In [ ]:
# Dica: modelo.conv1.weight.requires_grad
resposta = None

conferir(1, resposta)

### Exercício 2. O bloco que soltou

Todos os parâmetros de `layer4` estão livres para treinar? Responda `True` ou `False`.

<details>
<summary>Ver solução</summary>

```python
resposta = all(p.requires_grad for p in modelo.layer4.parameters())
```

</details>

In [ ]:
# Dica: all(p.requires_grad for p in modelo.layer4.parameters())
resposta = None

conferir(2, resposta)

### Exercício 3. Dois grupos

Quantos grupos de taxa de aprendizado o `otimizador` tem?

<details>
<summary>Ver solução</summary>

```python
resposta = len(otimizador.param_groups)
```

</details>

In [ ]:
# Dica: otimizador.param_groups é a lista. Cada grupo tem o seu lr.
resposta = None

conferir(3, resposta)

### Exercício 4. O passo da base

Qual é a taxa de aprendizado do primeiro grupo, o da base que soltou?

<details>
<summary>Ver solução</summary>

```python
resposta = otimizador.param_groups[0]['lr']
```

</details>

In [ ]:
# Dica: otimizador.param_groups[0]['lr']
resposta = None

conferir(4, resposta)

### Exercício 5. O passo da cabeça

Qual é a taxa de aprendizado do segundo grupo, o da cabeça nova?

<details>
<summary>Ver solução</summary>

```python
resposta = otimizador.param_groups[1]['lr']
```

</details>

In [ ]:
# Dica: O índice 1 é a cabeça.
resposta = None

conferir(5, resposta)

### Exercício 6. Pesos do bloco

Quantos parâmetros tem o `layer4`, livres e parados somados?

<details>
<summary>Ver solução</summary>

```python
resposta = contar(modelo.layer4)
```

</details>

In [ ]:
# Dica: contar(modelo.layer4)
resposta = None

conferir(6, resposta)

In [ ]:
placar()

## O que fica

Soltar não é soltar tudo. `conv1` segue parada, `layer4` volta, e as duas taxas convivem no mesmo `Adam`. Isso ainda não é o treino nas suas fotos. É o freio e o acelerador, prontos antes de andar.

Os exercícios usam `modelo`, `otimizador` e `contar`.